# 01 — Data Understanding

Load the raw dataset, inspect its structure, assess data quality, and understand the target distribution.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "train.csv")
test  = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "test.csv")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")

## 1. Sample

In [ ]:
train.head()

## 2. Schema

In [ ]:
descriptions = {
    "id": "Unique customer identifier",
    "Gender": "Male / Female",
    "Age": "Customer age in years",
    "Driving_License": "1 = has driving licence",
    "Region_Code": "Encoded region of residence",
    "Previously_Insured": "1 = already has vehicle insurance",
    "Vehicle_Age": "Age category of the vehicle",
    "Vehicle_Damage": "Yes/No — history of damage",
    "Annual_Premium": "Annual health insurance premium (USD)",
    "Policy_Sales_Channel": "Anonymised sales channel code",
    "Vintage": "Days since joining the company",
    "Response": "Target — 1 = interested in vehicle insurance",
}

schema = pd.DataFrame({
    "column": train.columns,
    "dtype": train.dtypes.values,
    "missing_pct": (train.isna().mean() * 100).round(2).values,
    "nunique": train.nunique().values,
    "description": [descriptions.get(c, "") for c in train.columns],
})
schema

## 3. Missing Values

In [ ]:
missing = train.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")

## 4. Target Distribution

In [ ]:
counts = train["Response"].value_counts()
pcts   = train["Response"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Not interested (0)", "Interested (1)"], counts.values, color=["#5b8db8", "#e07b39"])
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=11)
ax.set_title("Target Distribution — Response", fontsize=13)
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES / "01_target_distribution.png", dpi=150)
plt.show()

print(f"Positive rate: {pcts[1]:.2f}%")

## 5. Descriptive Statistics

In [ ]:
train.describe(include="all").T

## 6. Key Observations

- The dataset has **no missing values** — no imputation is required during EDA, although the pipeline includes it for robustness.
- The target is highly **imbalanced** (~12 % positive). We must use stratified splits and metrics like AP and lift rather than raw accuracy.
- `Vehicle_Age` and `Vehicle_Damage` are string-type and need encoding.
- `Region_Code` and `Policy_Sales_Channel` are numeric codes — could be treated as categorical or numeric; we will treat them as numeric.
- `Driving_License` has near-zero variance (almost everyone has a licence) — it may be dropped or kept.